In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
df = spark.table("databriks.bronze.products")
display(df)

In [0]:
df.groupBy("product_id").count().where("count(product_id) > 1").show()

In [0]:
display(df.select("category_code").distinct())

In [0]:
df = df.withColumn("category_code",F.upper("category_code"))
display(df.select("category_code").distinct())

In [0]:
display(df.select("brand_code").distinct())

In [0]:
df = df.withColumn("brand_code",F.upper("brand_code"))
display(df)

In [0]:
df = df.withColumn("weight_grams", F.regexp_replace("weight_grams", "g", ""))
display(df)

In [0]:
df = df.withColumn("width_cm", F.regexp_replace("width_cm", r"\.", ",")).withColumn("height_cm", F.regexp_replace("height_cm", r"\.", ","))
display(df)

In [0]:
# Fix spelling mistakes
df = df.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)
df.select("material").distinct().show()    

In [0]:
df = df.withColumn("rating_count",F.when(F.col("rating_count").isNull(), 0).otherwise(F.abs(F.col("rating_count"))))
display(df)

In [0]:
columns = df.columns
for col in columns:
    print(f"{col} has: {df.filter(F.col(col).isNull()).count()} null value")

In [0]:
df = df.withColumn("color", F.when(F.col("color").isNull(), F.lit("Not Available")).otherwise(F.col("color")))
display(df.select("color").distinct())

In [0]:
df.write.format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable("databriks.silver.products")